In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType, DoubleType

bronze_table = (spark.read.table("retail_sales_dev.bronze.exchange_rates_raw"))

flat_df = (
    bronze_table
    .withColumn("rates_json",F.to_json("rates"))
    .withColumn(
        "rates_map",
        F.from_json("rates_json", MapType(StringType(),MapType(StringType(),DoubleType())))
    )
    .select("base", F.explode("rates_map").alias("RateDate","currency_map"))
    .select("base","RateDate",F.explode("currency_map").alias("TargetCurrency","Rate"))
    .withColumnRenamed("base","BaseCurrency")
    .withColumn("RateDate",F.to_date("RateDate"))
)

# flat_df.show(10)

# # flat_df.filter(F.col("TargetCurrency").isNull()).count()
# df= flat_df.groupBy("RateDate").count().filter("count < 3")

# display(df)

In [0]:
(
    flat_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_sales_dev.silver.exchange_rates_cleaned")
)